### Set initial coordinates for study in sixtrack (aka create fort.13)

- Here set the initial coordinates that were used on a study in pysixtrack. Aka create the fort.13 file. Note : this script doesn't create a disribution. It just initialize particles given their 6D coordinates with respect to closed orbit. For creating a distribution check Sofia's sxript at: https://swan006.cern.ch/user/natriant/share/skostogl/distributions_Sixtrack/create_distribution

- Requirements: pysixtrack and cobjects. They can be installed using : pip install

In [2]:
import numpy as np
from math import *
import matplotlib.pyplot as plt
import pandas as pd
import pysixtrack

### 1. Load the coordinates of the particle from the pysixtrack study
In this example you load the coordinates of 5 particles with respect to closed orbit


In [3]:
df_2 = pd.read_pickle('/afs/cern.ch/work/n/natriant/private/pysixtrack_example_crabcavity/input/initial_distribution_wrt_CO.pkl') 

In [4]:
df_2

{'Dx_wrt_CO': array([ 0.00111371, -0.00182083,  0.00051376,  0.00085057, -0.0017093 ]),
 'Dy_wrt_CO': array([-1.70284716e-07,  2.82651067e-07, -3.24831527e-06, -1.22535120e-06,
        -2.06993879e-07]),
 'Dpx_wrt_CO': array([ 4.43339785e-05, -6.77386667e-05, -3.09591128e-05, -5.47579276e-05,
        -2.80533452e-05]),
 'Dpy_wrt_CO': array([-1.12031909e-07,  6.50703807e-08,  7.72992263e-08,  2.70686717e-08,
         1.17020180e-08]),
 'Dsigma_wrt_CO': array([ 0.09192294, -0.09597356, -0.04812763, -0.12204841,  0.02894211]),
 'Ddelta_wrt_CO': array([-0.00043418, -0.00069473, -0.00198029,  0.00099695,  0.00099689])}

### For the tracking using sixtrack you need at least a pair of particles. 
That's why you keep the coordinates of the two first particles

In [5]:
Dx_wrt_CO_m = df_2['Dx_wrt_CO'][:2]
Dpx_wrt_CO_rad = df_2['Dpx_wrt_CO'][:2]
Dy_wrt_CO_m = df_2['Dy_wrt_CO'][:2]
Dpy_wrt_CO_rad = df_2['Dpy_wrt_CO'][:2]
Dsigma_wrt_CO_m = df_2['Dsigma_wrt_CO'][:2]
Ddelta_wrt_CO  =  df_2['Ddelta_wrt_CO'][:2]

### Load the coordinates of the particle on the closed orbit, from pysixtrack

In [11]:
df_CO = pd.read_pickle('/afs/cern.ch/work/n/natriant/private/pysixtrack_example_crabcavity/input/particle_on_CO.pkl') 

### 2. Create fort.13
As in function  track_particle_sixtrack() in :
https://github.com/kparasch/sixtracklib_tracking_lhc/blob/1555cb1a3acd83c3027718f00cf53377fa2d86ee/helpers.py

You need at least two particles, (always multiple of 2)

In [21]:
n_part = len(Dx_wrt_CO_m)

In [22]:
lines_f13 = []

for i_part in range(n_part):
    temp_part = pysixtrack.Particles(**df_CO)
    temp_part.x     += Dx_wrt_CO_m[i_part]
    temp_part.px    += Dpx_wrt_CO_rad[i_part]
    temp_part.y     += Dy_wrt_CO_m[i_part]
    temp_part.py    += Dpy_wrt_CO_rad[i_part]
    temp_part.sigma += Dsigma_wrt_CO_m[i_part]
    temp_part.delta += Ddelta_wrt_CO[i_part]

    lines_f13.append('%.10e\n' % ((temp_part.x) * 1e3))
    lines_f13.append('%.10e\n' % ((temp_part.px) * temp_part.rpp * 1e3))
    lines_f13.append('%.10e\n' % ((temp_part.y) * 1e3))
    lines_f13.append('%.10e\n' % ((temp_part.py) * temp_part.rpp * 1e3))
    lines_f13.append('%.10e\n' % ((temp_part.sigma) * 1e3))
    lines_f13.append('%.10e\n' % ((temp_part.delta)))
    if i_part % 2 == 1:
        lines_f13.append('%.10e\n' % (temp_part.energy0 * 1e-6))
        lines_f13.append('%.10e\n' % (prev_part.energy * 1e-6))
        lines_f13.append('%.10e\n' % (temp_part.energy * 1e-6))
    prev_part = temp_part

In [23]:
with open('fort.13', 'w') as fid:
    fid.writelines(lines_f13)